# Aula 02 - Explorando o Microsoft Agent Framework

O **Microsoft Agent Framework (MAF)** é um framework unificado para construir agentes de IA. Fornece uma arquitetura limpa e compositável com quatro blocos básicos:

- **Cliente** – conecta-se a um endpoint de modelo de IA e gere a comunicação
- **Agente** – envolve um cliente com instruções e definições de ferramentas
- **Ferramentas** – expandem as capacidades do agente com funções personalizadas que o modelo pode chamar
- **Sessão** – mantém o histórico de conversas para interações de múltiplas voltas

Nesta aula, vamos construir um **agente de reservas de viagens** que verifica a disponibilidade do destino usando estes conceitos.


## Configuração


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## Compreender a Arquitetura do Framework Agent

O Microsoft Agent Framework segue uma arquitetura em camadas:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Cliente** – Um `FoundryChatClient` liga-se a uma implementação Azure OpenAI. Gere a autenticação, formatação de pedidos e análise de respostas.
2. **Agente** – Criado a partir do cliente via `provider.create_agent()`, o agente combina o acesso ao modelo com instruções (prompt do sistema) e ferramentas.
3. **Ferramentas** – Funções Python decoradas com `@tool` que o agente pode invocar para executar ações ou obter dados.
4. **Sessão** – Um objeto `AgentSession` (criado via `agent.create_session()`) que guarda o histórico da conversa, permitindo diálogo multi-turno onde o agente lembra o contexto anterior.

Vamos construir cada camada passo a passo.


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## Adicionar Ferramentas com o Decorador @tool

As ferramentas permitem que os agentes executem ações para além de gerar texto. O decorador `@tool` converte uma função Python normal numa coisa que o agente pode chamar.

Pontos chave:
- Use `Annotated[type, "description"]` para que o modelo compreenda cada parâmetro.
- A docstring torna-se a descrição da ferramenta que o modelo vê.
- `approval_mode="never_require"` significa que a ferramenta é executada automaticamente sem confirmação do utilizador.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Criar um Agente com Ferramentas

Agora combinamos o cliente, as instruções e as ferramentas num agente. As `instructions` atuam como o prompt do sistema — definem a persona e o comportamento do agente.


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Conversas de Várias Voltas com Sessões

Uma `AgentSession` (criada através de `agent.create_session()`) mantém o registo de todas as mensagens numa conversa. Ao passar a mesma sessão para cada chamada `agent.run()`, o agente tem acesso a todo o histórico da conversa e pode referir-se a mensagens anteriores.

Passamos `tools=[check_destination_availability]` para que o agente possa utilizar o nosso verificador de disponibilidade durante cada volta.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Resumo

Nesta lição explorou os quatro pilares do Microsoft Agent Framework:

| Conceito | O que Aprendeu |
|---------|------------------|
| **Cliente** | `FoundryChatClient` liga-se ao Azure OpenAI com autenticação baseada em credenciais |
| **Agente** | `provider.create_agent()` combina uma ligação a um modelo com instruções e um nome |
| **Ferramentas** | O decorador `@tool` expõe funções Python para o agente chamar |
| **Sessão** | `agent.create_session()` mantém o histórico da conversa ao longo de múltiplas interações |

Estes blocos constituintes combinam-se para criar agentes que podem manter conversas naturais, chamar funções externas e manter contexto — a base para padrões mais avançados de agentes em lições futuras.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:
Este documento foi traduzido utilizando o serviço de tradução automática [Co-op Translator](https://github.com/Azure/co-op-translator). Embora nos esforcemos pela precisão, esteja ciente de que traduções automáticas podem conter erros ou imprecisões. O documento original na sua língua nativa deve ser considerado a fonte autorizada. Para informações críticas, recomenda-se tradução profissional humana. Não nos responsabilizamos por quaisquer mal-entendidos ou interpretações incorretas resultantes da utilização desta tradução.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
